# Atelier 1 — Explorer les sources de données
## Projet fil rouge : BigData Commerce Analytics

**Niveau :** B2 · **Modalité :** binôme · 

### Mission
Vous êtes une équipe Data chargée d'étudier les sources d'une plateforme e-commerce de démonstration. Votre rôle n'est pas encore de construire un modèle : vous devez comprendre quelles données existent, qui les produit, comment elles sont structurées, quelles limites elles présentent et quelle valeur métier elles peuvent apporter.

### Compétences visées
- Interroger une API REST et comprendre une réponse JSON.
- Distinguer source, ressource, enregistrement et format.
- Inventorier les sources et leurs métadonnées.
- Justifier qualitativement les 5V.
- Identifier les usages métier et les risques de qualité.

> **Précaution de méthode :** DummyJSON fournit des données fictives/de démonstration. Les paniers ne constituent pas un historique de ventes réel. Les ressources et leurs volumes peuvent évoluer. Les résultats de votre collecte sont un instantané.


## 0. Préparer l'environnement

Exécutez cette cellule. Si `requests` n'est pas installé, lancez dans un terminal : `pip install requests pandas matplotlib`.


In [ ]:
from pathlib import Path
import json, requests, pandas as pd
BASE = "https://dummyjson.com"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
print("Dossier projet :", ROOT)
print("Dossier données brutes :", RAW)


## 1. Comprendre une API REST

Une API expose des ressources accessibles par des URL. Une requête GET demande une représentation de la ressource. DummyJSON renvoie généralement un objet JSON contenant une liste, un total, un décalage (`skip`) et une limite (`limit`).

Exemple : `/products?limit=0` demande tous les produits disponibles au moment de l'appel.


In [ ]:
url = f"{BASE}/products?limit=5"
response = requests.get(url, timeout=30)
response.raise_for_status()
payload = response.json()
print("Code HTTP :", response.status_code)
print("Clés de premier niveau :", list(payload.keys()))
print("Total annoncé par l'API :", payload.get("total"))
print("Nombre reçu :", len(payload.get("products", [])))
display(pd.json_normalize(payload["products"]).head())


### Questions de compréhension
1. Que signifie le code HTTP 200 ?
2. Pourquoi l'API renvoie-t-elle `total`, `skip` et `limit` ?
3. Quelle différence faites-vous entre une API et un fichier CSV ?
4. Pourquoi faut-il conserver la date de collecte d'un instantané ?


In [ ]:
# À compléter par le binôme
reponses_api = {
    "http_200": "",
    "pagination": "",
    "api_vs_csv": "",
    "date_collecte": ""
}
reponses_api


## 2. Inventorier les ressources disponibles

Le script de collecte du kit interroge plusieurs ressources : produits, paniers, utilisateurs, publications, commentaires, recettes, citations et tâches. Les trois premières sont centrales pour le scénario e-commerce ; les autres permettent d'élargir la réflexion sur la variété des données.


In [ ]:
resources = ["products", "carts", "users", "posts", "comments", "recipes", "quotes", "todos"]
inventaire = []
mode_donnees = "API en direct"
for resource in resources:
    try:
        r = requests.get(f"{BASE}/{resource}?limit=0", timeout=15)
        r.raise_for_status()
        p = r.json()
        count = len(p.get(resource, [])) if isinstance(p, dict) else len(p)
        inventaire.append({"ressource":resource, "mode":mode_donnees, "code_http":r.status_code,
                           "total_api":p.get("total", count), "elements_reçus":count,
                           "clés_réponse":", ".join(p.keys()) if isinstance(p, dict) else "liste"})
    except Exception:
        # Repli sur l'échantillon local fourni dans data/raw/
        mode_donnees = "échantillon local hors ligne"
        local_file = RAW / f"{resource}.json"
        if local_file.exists():
            p = json.loads(local_file.read_text(encoding="utf-8"))
            count = len(p.get(resource, [])) if isinstance(p, dict) else len(p)
            inventaire.append({"ressource":resource, "mode":mode_donnees, "code_http":"local",
                               "total_api":p.get("total", count), "elements_reçus":count,
                               "clés_réponse":", ".join(p.keys()) if isinstance(p, dict) else "liste"})
        else:
            inventaire.append({"ressource":resource, "mode":"indisponible", "code_http":"ERREUR",
                               "total_api":None, "elements_reçus":None, "clés_réponse":"pas de fichier local"})
inventaire_df = pd.DataFrame(inventaire)
display(inventaire_df)
print("Attention : si le mode local est utilisé, les volumes sont ceux du mini-échantillon pédagogique.")


### Travail demandé
- Relevez les trois ressources les plus utiles pour analyser l'activité e-commerce.
- Expliquez pourquoi `posts` et `comments` ne sont pas des commandes.
- Comparez le volume des ressources : laquelle est la plus volumineuse dans votre collecte ?
- Notez la date et l'heure de votre collecte.


In [ ]:
from datetime import datetime, timezone
date_collecte = datetime.now(timezone.utc).isoformat(timespec="seconds")
print("Collecte effectuée le :", date_collecte)


## 3. Examiner la structure des données

Une donnée JSON peut contenir des objets imbriqués et des tableaux. Cette variété est importante : les informations d'un panier, par exemple, sont imbriquées dans une liste `products`.


In [ ]:
def charger_ressource(resource):
    try:
        r = requests.get(f"{BASE}/{resource}?limit=0", timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception:
        local_file = RAW / f"{resource}.json"
        if not local_file.exists():
            raise FileNotFoundError(f"API inaccessible et fichier local absent : {local_file}")
        print(f"API inaccessible : utilisation de l'échantillon local {resource}.json")
        return json.loads(local_file.read_text(encoding="utf-8"))

products_payload = charger_ressource("products")
carts_payload = charger_ressource("carts")
users_payload = charger_ressource("users")

products_df = pd.json_normalize(products_payload["products"])
carts_df = pd.json_normalize(carts_payload["carts"])
users_df = pd.json_normalize(users_payload["users"])

print("Produits :", products_df.shape)
print("Paniers :", carts_df.shape)
print("Utilisateurs :", users_df.shape)
display(products_df.head(3))
display(carts_df.head(3))
display(users_df.head(3))


### Questions d'analyse
1. Quels champs identifient un produit de manière unique ?
2. Quel champ relie un panier à un utilisateur ?
3. Pourquoi la colonne `products` d'un panier est-elle complexe à analyser dans un tableau plat ?
4. Quel est le rôle d'une opération de normalisation (`json_normalize`) ?


In [ ]:
# Explorer les champs et les types
print("Champs produits :", products_df.columns.tolist())
print("\nTypes produits :")
display(products_df.dtypes.astype(str).to_frame("type"))
print("\nStructure d'un panier :")
print(json.dumps(carts_payload["carts"][0], ensure_ascii=False, indent=2)[:3000])


## 4. Construire la fiche d'identité des sources

Complétez le tableau avec des éléments observés. Pour la fréquence, DummyJSON ne doit pas être interprété comme un flux de transactions en direct : vous interrogez une API de démonstration à la demande.


In [ ]:
fiche_sources = pd.DataFrame([
    {"source":"Produits", "endpoint":"/products", "producteur":"DummyJSON", "format":"JSON", "volume_observe":len(products_df), "frequence":"à la demande", "usage_metier":"catalogue, prix, catégories", "limite":"données fictives, stock non transactionnel"},
    {"source":"Paniers", "endpoint":"/carts", "producteur":"DummyJSON", "format":"JSON imbriqué", "volume_observe":len(carts_df), "frequence":"à la demande", "usage_metier":"analyse de panier et quantités", "limite":"pas un journal réel de commandes"},
    {"source":"Utilisateurs", "endpoint":"/users", "producteur":"DummyJSON", "format":"JSON avec objets imbriqués", "volume_observe":len(users_df), "frequence":"à la demande", "usage_metier":"segmentation pédagogique", "limite":"profils fictifs ; minimiser les champs"},
    {"source":"Avis / commentaires", "endpoint":"/comments", "producteur":"DummyJSON", "format":"JSON", "volume_observe":"à mesurer", "frequence":"à la demande", "usage_metier":"analyse de texte", "limite":"commentaires associés aux posts, pas aux produits"}
])
display(fiche_sources)


## 5. Évaluer les 5V

Pour chaque V, attribuez une appréciation **faible / moyen / élevé** et justifiez-la avec un fait observé. Ne donnez pas une appréciation sans argument.

- **Volume :** nombre d'enregistrements et taille des fichiers.
- **Vitesse :** fréquence de génération ou de mise à jour. Une API accessible ne signifie pas que les données sont du temps réel.
- **Variété :** ressources, formats, champs simples, objets et tableaux imbriqués.
- **Véracité :** cohérence, valeurs absentes, doublons, valeurs aberrantes et caractère fictif.
- **Valeur :** décisions que les données pourraient éclairer.


In [ ]:
analyse_5v = pd.DataFrame([
    {"V":"Volume","niveau":"","preuve_chiffree":"","justification":""},
    {"V":"Vitesse","niveau":"","preuve_chiffree":"","justification":""},
    {"V":"Variété","niveau":"","preuve_chiffree":"","justification":""},
    {"V":"Véracité","niveau":"","preuve_chiffree":"","justification":""},
    {"V":"Valeur","niveau":"","preuve_chiffree":"","justification":""},
])
display(analyse_5v)


## 6. Usages métier, limites et risques

Pour chaque source principale, proposez au moins deux usages et deux limites. Exemples d'usages : analyser les catégories, repérer les paniers volumineux, comprendre la répartition des prix, préparer une segmentation illustrative. Exemples de limites : données fictives, absence d'horodatage transactionnel fiable, absence de paiements réels, volumes limités, absence de relation directe entre commentaires et produits.


In [ ]:
usages_limites = pd.DataFrame([
    {"source":"Produits","usage_1":"","usage_2":"","limite_1":"","limite_2":""},
    {"source":"Paniers","usage_1":"","usage_2":"","limite_1":"","limite_2":""},
    {"source":"Utilisateurs","usage_1":"","usage_2":"","limite_1":"","limite_2":""},
    {"source":"Commentaires","usage_1":"","usage_2":"","limite_1":"","limite_2":""}
])
display(usages_limites)


## 7. Synthèse et livrable

### À remettre par binôme
1. Le tableau d'inventaire des ressources.
2. La fiche d'identité de trois sources au minimum.
3. L'évaluation argumentée des 5V.
4. Deux usages et deux limites pour chaque source principale.
5. Une conclusion de 8 à 12 lignes : quelles sources conserveriez-vous pour un projet d'analyse e-commerce et pourquoi ?

### Restitution orale
3 minutes par binôme. Présentez une source, un V dominant, une opportunité métier et une limite importante.

### Critères de réussite
- Données observées et chiffres sourcés dans la collecte.
- Distinction claire entre données API, données dérivées et hypothèses.
- Justifications argumentées, pas de simples mots-clés.
- Prise en compte du caractère fictif de DummyJSON.

**Transition vers l'Atelier 2 :** vous allez profiler un fichier tabulaire de lignes de panier avec Pandas et examiner les problèmes de qualité.
